# 06 · Regression: OLS, Ridge and leave-one-protein-out validation

Goes beyond single-position correlations and asks whether the **14 positional embedding distances
together** predict the experimental change in affinity (report, section 4).

* **Features `X`:** local embedding distance of the mutant to the WT at positions 2–15
  (`pos_2 … pos_15`, 14 features) – from notebook 03.
* **Target `y`:** ΔZ-score of the Pm-Rb construct (`delta_pm_rb`) – from notebook 01.
* **Models:** ordinary least squares (OLS) and Ridge (L2) regression, both on standardised `X` and `y`.
* **Validation:** *leave-one-protein-out* (LOGO) over the three human proteins – train on two,
  test on the third, so the test protein is never seen during training.
  `StandardScaler`s (for `X` and, through `TransformedTargetRegressor`, for `y`) and the Ridge
  penalty (`RidgeCV`, α ∈ [1e-3, 1e10]) are fitted **on the training proteins only**
  (no data leakage).

**Inputs:** `data/processed/df_dms.csv`, `data/processed/mutant_embedding_distances.csv`.
**Outputs:** `results/logo_metrics.csv`, `results/logo_coefficients.csv`, ... and figures.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# Project paths
#   * Locally: start Jupyter from the `notebooks/` folder.
#   * Google Colab: Google Drive is mounted; edit the folder name below so it
#     points to the project folder stored in your Drive.
#   * Anywhere: set the PROJECT_DIR environment variable to override.
# ---------------------------------------------------------------------------
try:
    from google.colab import drive

    drive.mount("/content/drive")
    DEFAULT_PROJECT_DIR = Path("/content/drive/MyDrive/plm-embeddings-lxcxe-prb")
except ImportError:
    DEFAULT_PROJECT_DIR = Path.cwd().parent

PROJECT_DIR = Path(os.environ.get("PROJECT_DIR", DEFAULT_PROJECT_DIR))
RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
RESULTS_DIR = PROJECT_DIR / "results"
FIGURES_DIR = PROJECT_DIR / "figures"
for folder in (PROCESSED_DIR, RESULTS_DIR, FIGURES_DIR):
    folder.mkdir(parents=True, exist_ok=True)


def save_figure(name):
    """Save the current matplotlib figure into figures/ (call before plt.show())."""
    plt.savefig(FIGURES_DIR / name, dpi=300, bbox_inches="tight")

import seaborn as sns
from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import LinearRegression, Ridge, RidgeCV
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

## 1. Load and merge

In [ ]:
df_dms = pd.read_csv(PROCESSED_DIR / "df_dms.csv")
df_distances = pd.read_csv(PROCESSED_DIR / "mutant_embedding_distances.csv")

df_result = df_distances.merge(
    df_dms[["protein", "mutation", "delta_pm_rb"]],
    on=["protein", "mutation"],
    how="left",
    validate="many_to_one",
)
assert df_result["delta_pm_rb"].notna().all(), "Some mutations have no experimental ΔZ"

TARGET = "delta_pm_rb"
FEATURE_COLS = [f"pos_{i}" for i in range(2, 16)]  # mutated positions 2-15

LOGO_PROTEINS = ["ARI4A_HUMAN", "KDM5A_HUMAN", "HDAC1_HUMAN"]  # human proteins = the 3 folds
LOGO_MODELS = ["ESM-2 (650M)", "ProtT5"]
RIDGE_ALPHAS = np.logspace(-3, 10, 500)

## 2. Leave-one-protein-out cross-validation

The same routine evaluates any scikit-learn regressor built by a factory function.

In [ ]:
def make_ols():
    """Standardised OLS; y is standardised inside the model (fitted on training data only)."""
    pipeline = Pipeline([("scaler", StandardScaler()), ("model", LinearRegression())])
    return TransformedTargetRegressor(regressor=pipeline, transformer=StandardScaler())


def make_ridge():
    """Standardised Ridge with the penalty chosen by internal 5-fold cross-validation."""
    ridge = RidgeCV(alphas=RIDGE_ALPHAS, cv=5, scoring="neg_mean_squared_error")
    pipeline = Pipeline([("scaler", StandardScaler()), ("model", ridge)])
    return TransformedTargetRegressor(regressor=pipeline, transformer=StandardScaler())


def run_logo_cv(factory, regression_name, models=LOGO_MODELS, proteins=LOGO_PROTEINS):
    """Leave-one-protein-out CV. Returns (metrics table, coefficients table)."""
    metrics, coefficients = [], []
    for model_name in models:
        df_model = df_result[
            (df_result["embedding_model"] == model_name) & (df_result["protein"].isin(proteins))
        ]
        for fold, test_protein in enumerate(proteins, start=1):
            train_proteins = [p for p in proteins if p != test_protein]
            train = df_model[df_model["protein"].isin(train_proteins)]
            test = df_model[df_model["protein"] == test_protein]

            estimator = factory()
            estimator.fit(train[FEATURE_COLS].values, train[TARGET].values)
            pred_train = estimator.predict(train[FEATURE_COLS].values)
            pred_test = estimator.predict(test[FEATURE_COLS].values)

            fitted = estimator.regressor_.named_steps["model"]
            metrics.append({
                "regression": regression_name,
                "embedding_model": model_name,
                "fold": fold,
                "test_protein": test_protein,
                "train_proteins": ", ".join(train_proteins),
                "best_alpha": getattr(fitted, "alpha_", np.nan),
                "mse_train": mean_squared_error(train[TARGET], pred_train),
                "r2_train": r2_score(train[TARGET], pred_train),
                "mse_test": mean_squared_error(test[TARGET], pred_test),
                "r2_test": r2_score(test[TARGET], pred_test),
                "intercept": fitted.intercept_,
            })
            for feature, beta in zip(FEATURE_COLS, fitted.coef_):
                coefficients.append({
                    "regression": regression_name,
                    "embedding_model": model_name,
                    "test_protein": test_protein,
                    "position": feature,
                    "coefficient": beta,
                })
    return pd.DataFrame(metrics), pd.DataFrame(coefficients)

### 2.1 OLS

In [ ]:
metrics_ols, coefs_ols = run_logo_cv(make_ols, "OLS")
metrics_ols[["embedding_model", "test_protein", "mse_test", "r2_test"]]

### 2.2 Ridge (L2)

In [ ]:
metrics_ridge, coefs_ridge = run_logo_cv(make_ridge, "Ridge")
metrics_ridge[["embedding_model", "test_protein", "best_alpha", "mse_test", "r2_test"]]

In [ ]:
df_metrics = pd.concat([metrics_ols, metrics_ridge], ignore_index=True)
df_coefs = pd.concat([coefs_ols, coefs_ridge], ignore_index=True)
df_metrics.to_csv(RESULTS_DIR / "logo_metrics.csv", index=False)
df_coefs.to_csv(RESULTS_DIR / "logo_coefficients.csv", index=False)

# Mean over the three folds
(df_metrics.groupby(["embedding_model", "regression"])
    [["r2_train", "r2_test", "mse_train", "mse_test", "best_alpha"]]
    .mean().reset_index())

## 3. Coefficients per position

Standardised coefficients β for each fold (colour = test protein), report figures 7 and 8.
Core positions are 5, 7 and 9.

In [ ]:
def plot_coefficients(regression_name, model_name, save=True):
    sub = df_coefs[
        (df_coefs["regression"] == regression_name) & (df_coefs["embedding_model"] == model_name)
    ]
    plt.figure(figsize=(14, 6))
    sns.barplot(data=sub, x="position", y="coefficient", hue="test_protein", palette="tab10")
    plt.axhline(0, color="gray", linestyle="--", linewidth=0.8)
    plt.title(f"β coefficients per position — {regression_name} ({model_name})",
              fontsize=13, fontweight="bold")
    plt.xlabel("Position")
    plt.ylabel("Standardised coefficient β")
    plt.legend(title="Test protein", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    if save:
        slug = model_name.split(" ")[0].lower().replace("-", "")
        save_figure(f"coefficients_{regression_name.lower()}_{slug}.png")
    plt.show()


for regression_name in ("OLS", "Ridge"):
    for model_name in LOGO_MODELS:
        plot_coefficients(regression_name, model_name)

## 4. Supplementary analyses

### 4.1 Similarity of the Ridge coefficients across folds

Pearson correlation between the β vectors learned in the three LOGO folds: high values mean the
positional importance profile is stable regardless of the protein left out.

In [ ]:
for model_name in LOGO_MODELS:
    betas = (
        coefs_ridge[coefs_ridge["embedding_model"] == model_name]
        .pivot(index="test_protein", columns="position", values="coefficient")
        .loc[LOGO_PROTEINS, FEATURE_COLS]
    )
    plt.figure(figsize=(6, 5))
    sns.heatmap(betas.T.corr(method="pearson"), annot=True, fmt=".2f",
                cmap="coolwarm", vmin=-1, vmax=1, square=True)
    plt.title(f"Correlation of Ridge β between folds ({model_name})")
    plt.xlabel("Test protein of the fold")
    plt.ylabel("Test protein of the fold")
    plt.tight_layout()
    plt.show()

### 4.2 Within-protein baseline

For comparison, a Ridge model is trained and tested **within** each protein (random 80/20 split of its
mutations, `random_state=42`), for all five proteins and three embeddings. Unlike LOGO, test mutations
here come from the same protein as the training mutations, so it measures a much easier task.

In [ ]:
baseline_rows = []
for protein in df_result["protein"].unique():
    for model_name in df_result["embedding_model"].unique():
        sub = df_result[(df_result["protein"] == protein) & (df_result["embedding_model"] == model_name)]
        X_train, X_test, y_train, y_test = train_test_split(
            sub[FEATURE_COLS], sub[TARGET], test_size=0.2, random_state=42
        )
        scaler = StandardScaler().fit(X_train)
        ridge = RidgeCV(alphas=[0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0], cv=5)
        ridge.fit(scaler.transform(X_train), y_train)
        baseline_rows.append({
            "protein": protein,
            "embedding_model": model_name,
            "best_alpha": ridge.alpha_,
            "r2_train": r2_score(y_train, ridge.predict(scaler.transform(X_train))),
            "r2_test": r2_score(y_test, ridge.predict(scaler.transform(X_test))),
            "mse_test": mean_squared_error(y_test, ridge.predict(scaler.transform(X_test))),
        })

df_baseline = pd.DataFrame(baseline_rows).sort_values("r2_test", ascending=False)
df_baseline.to_csv(RESULTS_DIR / "within_protein_baseline.csv", index=False)
df_baseline